In [1]:
import psycopg2
from sentence_transformers import SentenceTransformer, CrossEncoder
from pymilvus import connections, Collection

# ===== DB CONFIG =====
DB_CONFIG = {
    "host": "aws-1-ap-northeast-2.pooler.supabase.com",  # 🔥 đổi lại
    "port": 6543,                          # 🔥 không dùng 6543 nữa
    "database": "postgres",
    "user": "postgres.ypkwqsbsunlvpoqdadbq",
    "password": "5$eAK8EV4S+gsKj",
    "sslmode": "require"
}

conn = psycopg2.connect(**DB_CONFIG)

# ===== MODELS =====
embed_model = SentenceTransformer("BAAI/bge-small-en")
reranker = CrossEncoder("BAAI/bge-reranker-base")

In [2]:
def detect_intent(query):
    q = query.lower()

    return {
        "has_location": any(k in q for k in ["gần", "near", "km"]),
        "has_price": any(k in q for k in ["rẻ", "cheap", "đắt"]),
        "has_time": any(k in q for k in ["sáng", "trưa", "tối"]),
    }

In [3]:
def rrf_fusion(dense, sparse, k=60):
    scores = {}

    for doc_id, rank in dense:
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank)

    for doc_id, rank in sparse:
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [doc_id for doc_id, _ in ranked]

In [4]:
def sql_filter(candidate_ids, lat=None, lng=None, price=None, time=None):
    if not candidate_ids:
        return []

    query = """
        SELECT r.id, r.name, r.price_level,
               r.latitude, r.longitude,
               (6371 * acos(
                   cos(radians(%s)) * cos(radians(latitude)) *
                   cos(radians(longitude) - radians(%s)) +
                   sin(radians(%s)) * sin(radians(latitude))
               )) AS distance
        FROM restaurants r
        WHERE r.id = ANY(%s)
    """

    params = [lat, lng, lat, candidate_ids]

    if price:
        query += " AND price_level <= %s"
        params.append(price)

    if time:
        query += " AND open_time_slot = %s"
        params.append(time)

    query += " ORDER BY distance LIMIT 20"

    with conn.cursor() as cur:
        cur.execute(query, params)
        return cur.fetchall()

In [5]:
def fetch_text_for_rerank(rows):
    def get_row_id(row):
        if isinstance(row, dict):
            return row.get("restaurant_id") or row.get("id")
        if isinstance(row, (list, tuple)):
            return row[0]
        return row

    def get_menu_id(row):
        if isinstance(row, dict):
            return row.get("menu_id")
        return None

    ids = [get_row_id(r) for r in rows]
    ids = [r for r in ids if r is not None]
    menu_ids = [get_menu_id(r) for r in rows]
    menu_ids = [m for m in menu_ids if m is not None]

    if not ids and not menu_ids:
        return {}

    with conn.cursor() as cur:
        if menu_ids:
            if ids:
                cur.execute("""
                    SELECT m.restaurant_id,
                           m.dish_name,
                           m.description
                    FROM menus m
                    WHERE m.id = ANY(%s)
                      AND m.restaurant_id = ANY(%s)
                """, (menu_ids, ids))
            else:
                cur.execute("""
                    SELECT m.restaurant_id,
                           m.dish_name,
                           m.description
                    FROM menus m
                    WHERE m.id = ANY(%s)
                """, (menu_ids,))
        else:
            cur.execute("""
                SELECT m.restaurant_id,
                       m.dish_name,
                       m.description
                FROM menus m
                WHERE m.restaurant_id = ANY(%s)
            """, (ids,))

        data = cur.fetchall()

    text_map = {}
    for r_id, dish, desc in data:
        text_map.setdefault(r_id, "")
        text_map[r_id] += f"{dish}. {desc}. "

    return text_map

In [6]:
def rerank(query, rows):
    if not rows:
        return []

    text_map = fetch_text_for_rerank(rows)
    print("🔍 Rerank candidates:", text_map)
    def get_row_id(row):
        if isinstance(row, dict):
            return row.get("restaurant_id") or row.get("id")
        if isinstance(row, (list, tuple)):
            return row[0]
        return row

    docs = []
    for r in rows:
        rid = get_row_id(r)
        menu_id = r.get("menu_id") if isinstance(r, dict) else None
        if rid is None:
            continue
        text = text_map.get(rid, "")
        docs.append({"id": rid, "text": text, "menu_id": menu_id})

    if not docs:
        return []

    pairs = [(query, d["text"]) for d in docs]
    scores = reranker.predict(pairs)

    for i, s in enumerate(scores):
        docs[i]["score"] = float(s)

    docs.sort(key=lambda x: x["score"], reverse=True)

    return docs

In [7]:
def clean_query_for_fts(query):
    stopwords = ["quán", "gần", "tôi", "ở", "có", "không"]

    tokens = query.lower().split()
    tokens = [t for t in tokens if t not in stopwords]

    return " ".join(tokens)

In [8]:
def sparse_search(query, slots, top_k=20):
    queries = []

    # 1. ưu tiên dish
    if slots.get("dish"):
        queries.append(slots["dish"])

    # 2. luôn có query fallback
    #queries.append(query)

    all_results = []

    for q in queries:
        q_clean = clean_query_for_fts(q)

        with conn.cursor() as cur:
            cur.execute("""
                SELECT restaurant_id,
                       MAX(ts_rank(tsv, websearch_to_tsquery(%s))) AS score
                FROM menus
                WHERE tsv @@ websearch_to_tsquery(%s)
                GROUP BY restaurant_id
                ORDER BY score DESC
                LIMIT %s
            """, (q_clean, q_clean, top_k))

            rows = cur.fetchall()
            all_results.extend(rows)

    # merge score (max)
    score_map = {}
    for rid, score in all_results:
        score_map[rid] = max(score_map.get(rid, 0), score)

    ranked = sorted(score_map.items(), key=lambda x: x[1], reverse=True)

    # convert → RRF format
    return [(rid, i) for i, (rid, _) in enumerate(ranked)]

In [9]:
def fallback_like_search(query, top_k=10):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT restaurant_id
            FROM menus
            WHERE dish_name ILIKE %s
            LIMIT %s
        """, (f"%{query}%", top_k))

        return [(row[0], i) for i, row in enumerate(cur.fetchall())]

In [10]:
def sparse_search_3(query, slots, top_k=20):
    queries = []

    if slots.get("dish"):
        queries.append(slots["dish"])

    all_results = []

    for q in queries:
        q_clean = clean_query_for_fts(q)

        with conn.cursor() as cur:
            cur.execute("""
                SELECT DISTINCT ON (restaurant_id)
                       id AS menu_id,
                       restaurant_id,
                       dish_name,
                       ts_rank(tsv, websearch_to_tsquery(%s)) AS score
                FROM menus
                WHERE tsv @@ websearch_to_tsquery(%s)
                ORDER BY restaurant_id, score DESC
                LIMIT %s
            """, (q_clean, q_clean, top_k))

            rows = cur.fetchall()
            all_results.extend(rows)

    # 🔥 merge theo restaurant_id
    result_map = {}

    for menu_id, rid, dish_name, score in all_results:
        if rid not in result_map or score > result_map[rid]["score"]:
            result_map[rid] = {
                "menu_id": menu_id,
                "restaurant_id": rid,
                "dish_name": dish_name,
                "score": score
            }

    # sort
    ranked = sorted(result_map.values(), key=lambda x: x["score"], reverse=True)

    return ranked

In [11]:
def dense_search_3(query_vec, sparse_results, top_k=20):
    if not sparse_results:
        candidate_ids = None
    elif isinstance(sparse_results[0], dict):
        candidate_ids = [r["restaurant_id"] for r in sparse_results]
    else:
        candidate_ids = list(sparse_results)

    query_vec = [float(x) for x in query_vec]

    with conn.cursor() as cur:
        if candidate_ids:
            cur.execute("""
                SELECT restaurant_id,
                       MIN(embedding <-> %s::vector) AS distance
                FROM menus
                WHERE restaurant_id = ANY(%s)
                GROUP BY restaurant_id
                ORDER BY distance ASC
                LIMIT %s
            """, (query_vec, candidate_ids, top_k))
        else:
            cur.execute("""
                SELECT restaurant_id,
                       MIN(embedding <-> %s::vector) AS distance
                FROM menus
                GROUP BY restaurant_id
                ORDER BY distance ASC
                LIMIT %s
            """, (query_vec, top_k))

        rows = cur.fetchall()

    return rows

In [12]:
def search_pipeline_v3(query, slots, lat=None, lng=None):
    print("="*50)
    print("🔍 QUERY:", query)
    print("🧩 SLOTS:", slots)

    conn.rollback()

    # 1. Embedding
    query_vec = embed_model.encode(query)

    # 2. Sparse → lấy món đại diện (menu-level)
    sparse_results = sparse_search_3(query, slots, top_k=50)

    print("🔍 Sparse candidates:", len(sparse_results))
    print("🍜 Top sparse dish:", sparse_results[:3])

    # 👉 FIX: lấy restaurant_id đúng format mới
    candidate_ids = [r["restaurant_id"] for r in sparse_results]

    # ⚠️ fallback nếu sparse yếu
    if len(candidate_ids) < 10:
        print("⚠️ Sparse yếu → dùng dense global")
        dense_results = dense_search_3(query_vec, None, top_k=50)
    else:
        dense_results = dense_search_3(query_vec, candidate_ids, top_k=20)

    print("⚡ Dense top5:", dense_results[:5])

    # 3. Merge: giữ dish từ sparse + rank từ dense
    dense_rank_map = {rid: rank for rank, (rid, _) in enumerate(dense_results)}

    merged = []
    for item in sparse_results:
        rid = item["restaurant_id"]

        if rid in dense_rank_map:
            item["dense_rank"] = dense_rank_map[rid]
            merged.append(item)

    # sort theo dense
    merged = sorted(merged, key=lambda x: x["dense_rank"])

    print("🔀 After merge:", len(merged))

    # 4. SQL Filter (restaurant-level)
    filtered_ids = sql_filter(
        [r["restaurant_id"] for r in merged],
        lat=lat,
        lng=lng,
        price=2 if slots.get("price") == "low" else None,
        time="evening" if slots.get("time") else None
    )

    print("📍 After SQL:", len(filtered_ids))

    # 👉 map lại full info (GIỮ dish)
    filtered_map = []
    for filter in filtered_ids:
        filtered_map.append(filter[0])
    print("filtered_map:", filtered_map)
    
    filtered = [r for r in merged if r["restaurant_id"] in filtered_map]
    for filter in filtered_ids:
        print("📍 Filtered restaurant_id:", filter[0])
    
    for r in merged:
        print(r["restaurant_id"], r["dish_name"], r["dense_rank"])
    print("filtered:", len(filtered))
    for r in filtered:
        print(r)
        print(r["restaurant_id"], r["dish_name"])
    # 5. Rerank (dùng dish làm context)
    reranked = rerank(query, filtered)
    print(type(reranked), len(reranked))
    
    top = reranked[:10]
    print("top:", top)
    print("\n🏆 FINAL RESULTS:")
    for r in top:
        print({
            "restaurant_id": r["id"],
            "dish": r["text"],
            "menu_id":r["menu_id"],
            "score": r["score"]
        })

    return top

In [13]:
slots = {
    "dish": "mì cay",
    "price": "low",
    "time": None,
    "location": "near"
}

search_pipeline_v3(
    query="quán mì cay gần tôi rẻ",
    slots=slots,
    lat=10.762622,
    lng=106.660172
)

🔍 QUERY: quán mì cay gần tôi rẻ
🧩 SLOTS: {'dish': 'mì cay', 'price': 'low', 'time': None, 'location': 'near'}
🔍 Sparse candidates: 20
🍜 Top sparse dish: [{'menu_id': 931, 'restaurant_id': 799, 'dish_name': 'Mì cay chay', 'score': 1.0}, {'menu_id': 1896, 'restaurant_id': 992, 'dish_name': 'Mì Cay 7 Cấp Độ Cực Hot', 'score': 0.999999}, {'menu_id': 2024, 'restaurant_id': 1017, 'dish_name': 'Bánh Khoai Mì Chiên Cay', 'score': 0.999977}]
⚡ Dense top5: [(686, 1.9798510790221), (1343, 2.11474975782917), (1182, 2.23152234635221), (641, 2.30134262205233), (992, 2.31618680123555)]
🔀 After merge: 20
📍 After SQL: 15
filtered_map: [741, 686, 799, 789, 773, 939, 955, 992, 1182, 968, 616, 1017, 641, 1083, 1314]
📍 Filtered restaurant_id: 741
📍 Filtered restaurant_id: 686
📍 Filtered restaurant_id: 799
📍 Filtered restaurant_id: 789
📍 Filtered restaurant_id: 773
📍 Filtered restaurant_id: 939
📍 Filtered restaurant_id: 955
📍 Filtered restaurant_id: 992
📍 Filtered restaurant_id: 1182
📍 Filtered restaurant_i

[{'id': 1017,
  'text': 'Bánh Khoai Mì Chiên Cay. Món bánh khoai mì chiên cay là món ăn vặt rất ngon miệng đấy.. ',
  'menu_id': 2024,
  'score': 0.007021140772849321},
 {'id': 799,
  'text': 'Mì cay chay. Mì cay là món ăn ngày càng phổ biến với giới trẻ và cả thực khách trung niên. Vậy hôm nay Món Ngon Mỗi Ngày sẽ giới thiệu đến bạn món Mì Cay Chay, giúp thực đơn nhà mình thêm màu sắc mới mẻ nhé. Với những nguyên liệu đơn giản như thịt bò chay, cá viên chay quyện cùng nước mì đậm đà thơm mùi ớt hiểm, ớt bột Hàn Quốc, hài hòa chua ngọt cay cùng sợi mì udon dẻo, thấm vị, món ngon này chắc hẳn sẽ khiến bữa chay của bạn thêm thú vị. ',
  'menu_id': 931,
  'score': 0.0054542128928005695},
 {'id': 773,
  'text': 'Mì Soba xào bạch tuộc. Bạch tuộc sừn sựt, giòn ngọt vừa chín tới, thấm vị tỏi phi thơm lừng. Mì soba cay nhẹ quyện lẫn khi ăn kèm càng khiến món ăn thêm đậm đà. Hãy chắc chắn là cơm đã được nấu đủ, bởi món ăn tuyệt ngon mang hương vị Nhật Bản này hứa hẹn sẽ đưa cơm lắm đấy. ',
  'm